# Datos geográficos de España

## Nomenclatura de Provincias y Comunidades Autónomas (INE)

Se usará la nomeclatura oficial del INE como nombres canónicos con finalidades de normalizar y armonizar los datos. Concretamente extraeremos el código y nombre de las comunidades autónomas y provincias de la tabla html de la fuente. Nótese que el código de cada provincia es el mismo que los 2 primeros dígitos de los códigos postales que pertenecen a cada provincia. 

Fuente: [web](https://www.ine.es/daco/daco42/codmun/cod_ccaa_provincia.htm)

In [ ]:
from src.scrapers.ine_simple_scraper import scrape_INE_ccaa_provinces

scrape_INE_ccaa_provinces()

[+] Fetching and parsing the webpage...
[+] Parsing HTML content...
[+] Extracting data from the table...


## Municipios y sus Provincia y Comunidad Autónoma relacionados

También vamos a extraer también el listado de todos los municipios del territorio nacional a partir del excel "diccionario25.xlsx" obtenido del [INE](https://www.ine.es/daco/daco42/codmun/diccionario25.xlsx).Dicha hoja de calculo excel contiene un recopilatorio de municipios actualizados a 1 de enero de 2025. Guardaremos los datos extraidos en un archivo de formato csv y con las columnas en minusculas y en snake_case.

In [ ]:
import pandas as pd
from src.shared.utils import normalize, slugify

municipios_df = pd.read_excel(
    "../data/raw/INE/diccionario25.xlsx",
    sheet_name="dic25",
    skiprows=1,
    usecols="A:E",
    dtype=str,
)

columns_dict = {}
for column in municipios_df.columns.tolist():
    columns_dict[column] = slugify(normalize(column)).lower()
municipios_df = municipios_df.rename(columns=columns_dict)

municipios_df.to_csv("../data/processed/municipios.csv", index=False)


## Codigos postales y municipios relacionados

También nos interesa conocer el listado de Códigos Postales que existen en el país. Nos servirá para identificar códigos erróneos y para el proceso de geocodificación(obtención de latitud y longitud).

Aunque el INE no nos proporciona de forma directa un listado de códigos postales nacionales, podemos extraerlo a partir del callejero censal. Este método se basa en el repositorio de [Iñigo Flores](https://github.com/inigoflores/ds-codigos-postales-ine-es).

El método consiste en:

1. Descargar el último archivo zip del callejero censal a través de [la página del INE](https://www.ine.es/dyngs/SER/es/index.htm?cid=1390). Se puede a través de una petición a la url: *www.ine.es/prodyser/callejero/caj_esp/caj_esp0{1 o 7}{4 dígitos año}.zip*. Ha fecha de 19 de enero de 2026 el archivo más recientes es del enero de 2025.
2. Buscar el archivo que contenga TRAM en el nombre dentro del archivo zip y extraer información de él.
3. Extraer de cada línea el código de municipio y el código postal medainte expresiones regulares y posiciones conocidas en la cadena.

Si bien el repositorio de Iñigo contiene un script con funcionalidades de extracción más avanzada, se ha preferido aislar el proceso de extracción únicamente el método anterior debido a que es suficiente para obtener un conjunto nacional de códigos postales y también para mantener la homogeneidad del stack tecnológico del proyecto(Python), ya que de haberse icorporado el script supondría la gestión de otro lengujae de programación(PHP) y sus dependencias.

Cabe destacar que no se incluirá en el repositorio el archivo .zip del callejero porque ocupa más de 500 MB de espacio de disco.

In [4]:
from src.processing.ine_parsers import parse_callejero_censal

callejero_df = parse_callejero_censal(month=1, year=2025)
callejero_df.to_csv("../data/silver/caj_esp_012025.csv", index=False)
print(f"códigos postales únicos recopilados: {callejero_df['codigo_postal'].nunique()}")

File already exists: ../data/bronze/INE/caj_esp_012025.zip
Skipping invalid line: 0809501001   000170100014...
Skipping invalid line: 4500602001   000170100228...
códigos postales únicos recopilados: 10824


# Datos de Empresas

## Startups certificadas por ENISA

Extraeremos de la incrustación del panel de PowerBI publicado en la fuente, las respuestas de las llamadas a la API de PowerBI.

Fuente: [web](https://www.enisa.es/es/certifica-tu-startup/startups)

In [ ]:
from src.scrapers.powerbi.queries import get_json_from_powerbi, enisa_certified_startups
from src.scrapers.powerbi.powerbi_parser import pbi_json_to_df
import json
data = get_json_from_powerbi(enisa_certified_startups())
data_df = pbi_json_to_df(data)
data_df = data_df.rename(
    columns={
        "razon_social": "denominacion",
        "cif": "nif",
        "DESCRIPCION": "comunidad_autonoma",
        "Provincia": "provincia",
        "FechaComité": "fecha_certificacion",
        "Fecha Estimada de Descertificación": "fecha_estimada_descertificacion",
        "Fecha de efecto de pérdida de certificación": "fecha_efecto_descertificacion",
    }
)
print(data_df.head())
data_df.to_csv("../data/bronze/ENISA/enisa_startups_certificadas.csv", index=False)

A partir de los nifs del Dataframe anterior podemos scrapear datos más detallados de las empresas a través de eInforma, Axesor o Iberinform.

In [ ]:
import pandas as pd
data_df = pd.read_csv("../data/bronze/ENISA/enisa_startups_certificadas.csv")
nifs = data_df["nif"].tolist()


Funciones auxiliares para facilitar el scraping a portales web

In [2]:
import os
import json
import pandas as pd
def aggregate_json_artifacts_into_csv(artifacts_directory, output_csv_path, output_filename, enconding='utf-8'):
    artifacts = [file for file in os.listdir(artifacts_directory)]
    data = []
    for artifact_file in artifacts:
        with open(os.path.join(artifacts_directory, artifact_file), 'r', encoding=enconding) as f:
            artifact = json.load(f)
            data.append(artifact)
    df = pd.DataFrame(data)
    if not os.path.exists(output_csv_path):
        os.makedirs(output_csv_path)
    output_csv = os.path.join(output_csv_path, output_filename)
    df.to_csv(output_csv, index=False)

def create_batches(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


### Scraper con eInforma

eIforma incorpora un sistema de rate limit por IP, se recomienda usar proxies o VPN para evitar bloqueos. El limite suele rondar unas 200 consultas diarias por IP.

In [ ]:
from src.scrapers.einforma_scraper import EinformaScraper
scraped = [file.split('.')[0] for file in os.listdir("../data/cache/einforma/")]
targets = [{"nif": nif} for nif in nifs if nif not in scraped]
batches = create_batches(targets, 200)
for batch in batches:
    scraper = EinformaScraper(headless=False, log_level='DEBUG')
    scraper.scrape(targets=batch, delay_interval=(4,6), timeout_xpath=10, export_artifact=True, artifact_path="../data/cache/einforma/")
    scraper.close()
#scraper.save_to_csv("../data/raw/einforma/einforma_startups_certificadas.csv")

aggregate_json_artifacts_into_csv(
    artifacts_directory="../data/cache/einforma/",
    output_csv_path="../data/bronze/einforma/",
    output_filename="einforma_startups_certificadas.csv",
    enconding='utf-8'
)

### Scraper con Iberinform

Iberinform no emplea rate limit por IP pero sí por sesión por ello se realiza reseteos del webdriver por batches. También incorpora CAPTCHAs para evitar el scraping, sin embargo no esta correctamente implementado y es posible hacer bypass de forma sencilla. Este es el scraper que ofrece mayor sencillez de extracción(no hay que rotar direcciones IP y se puede bypassear las protecciones) y mayor cantidad de información, se recomienda su uso pero es posible que contenga errores en la información extraida. Se debe revisar los datos obtenidos y contrastarlos con otras fuentes.

In [ ]:
from src.scrapers.iberinform_scraper import IberinformScraper
import os
scraped = [file.split('.')[0] for file in os.listdir("../data/cache/iberinform/")]
targets = [{"nif": nif} for nif in nifs if nif not in scraped]
batches = create_batches(targets, 80)
for batch in batches:
    scraper = IberinformScraper(headless=False)
    scraper.scrape(targets=batch, delay_interval=(4,6), timeout_xpath=10, export_artifact=True, artifact_path="../data/cache/iberinform/")
    scraper.close()

aggregate_json_artifacts_into_csv(
    artifacts_directory="../data/cache/iberinform/",
    output_csv_path="../data/bronze/iberinform/",
    output_filename="iberinform_startups_certificadas.csv"
)

### Scraper con Axesor
Axesor incorpora un sistema muy agresivo en detección de scraping, ha sido necesario emular comportamiento humano. Es el scraper más lento y que ofrece menor información. En la medida de lo posible se ha evitado su uso, se incluye solo a modo de referencia para futuras investigaciones.

In [ ]:
from src.scrapers.axesor_scraper import AxesorScraper
import os
scraped = [file.split('.')[0] for file in os.listdir("../data/cache/axesor/")]
scraper = AxesorScraper(headless=False,log_level='DEBUG')
targets = [{"nif": nif} for nif in nifs if nif not in scraped]
scraper.scrape(targets=targets, delay_interval=(5,8), timeout_xpath=10, export_artifact=True, artifact_path="../data/cache/axesor/")
#scraper.save_to_csv("../data/raw/axesor/axesor_startups_certificadas.csv")
aggregate_json_artifacts_into_csv(
    artifacts_directory="../data/cache/axesor/",
    output_csv_path="../data/bronze/axesor/",
    output_filename="axesor_startups_certificadas.csv",
    enconding='utf-8'
)

#### Scraper con el Registro Mercantil
Se ha podido semi-atomatizar la extracción de datos directamente desde el registro mercantil explotando peticiones a API con reutilización de IDs de captcha resueltos previamente.
Se obtiene respuestas json que en ocasiones faltan valores. Sorprendentemente los datos proporcionados por el registro mercantil puede haber carencias en información de actividad, de la empresa, domicilio, etc.

Con semi automatizar nos referimos que que previamente hay que resolver unos captcha de forma manual y guardar los id para ser reutilizados. Cada ID permite ser reutilizado para 100 consultas. Sin embargo cada id está ligada a una serie de cabeceras y cookies de sesión, por lo que no es posible automatizar completamente el proceso. No se recomienda scrapear los datos del Registro Mercantil solo se incluye como referencia para futuras investigaciones.

In [ ]:
#scraping con reutilización de ids de sesión y captchaId
from src.scrapers.registradores.datatableApi import search_by_term
from src.shared.utils import save_as_json
import os
import json
import time
scraped = [file.split('.')[0] for file in os.listdir("../data/cache/registradores/")]
# cockies y cabeceras de sesión obtenidos a través de la inspección de red en el navegador tras resolver un captcha de forma manual
JSESSIONID = '805B4A9A8344DEEFF19DA96E5142DEE7'
persistencia = '!gO0tfWAO+5AueEbie6XCJLX2iCB8dCq6GGumXTl66HSyZ9xrSHVPz/NmFnbRpR00Lk2lHxuPLqGoG74='
OClmoOot = 'Ay1is-qdAQAAvjLfrbGfZE5E3mtSNe9z5D5mYKF2LN8JCKcZuJnbQuatJfX1AU902pOucphhwH8AADQwAAAAAA|1|0|1850586ab04183f9bdd141f16dd6b637ecd2d713'
TS01dc4fc6 = '010e7b7cc2d6f75b9c93e06dab7997d82ae7501283ecff81ba5889b22405f6495b2545b696c56876645e4b8f0ef5fae6bf29021a52'
LFR_SESSION_STATE_20103 = '1777765642301'
TS01afd22c = '012ce6495c859bd502057b0f0186b05df6f9acf6b85bce723c530eb8dcb08d2c4a6e7e375c3cea0582c5a226f5ef3e988b84f0557c'
TS025671ed029 = '08ff354cadab280023ac5041bc8baca4ff4752bbdffd93582d1c6a15c5c456e00b7bcaebf5d0b56393d926b8a2df36d2'
TS56819fee027 = '08ff354cadab200014e280438a09f4c8d12695431b4ba21d16967d75860ff4bb634a5c08ee6d4d8808f4d35f1b113000ec2e5a4095c004df5d2378d260b3bf248adf2dd986a495d57d19059f4d7200bc4495b1f6ba6c518fbee2c3d3a5ab8c22'
# ids de captcha resueltos previamente, cada uno de ellos permite ser reutilizado para 100 consultas.
captchaIds = ['642e000ejvr7sfgqkbjp5nuug8','rhbk4l7boe2ta8tcklkj6ajvqq','8obng73srpa26ggnba3bdu37g0','rs33rum3c5ui3um79tricul8re'] # estos IDs estan caducados, se incluyen solo a modo de referencia para futuras investigaciones
captchaId = captchaIds.pop(0)
for nif in nifs:
    if nif in scraped:
        continue
    response = search_by_term(nif, JSESSIONID, persistencia, TS01dc4fc6, OClmoOot, LFR_SESSION_STATE_20103, TS56819fee027, TS01afd22c, captchaId)
    if response.status_code == 200:
        if response.json().get("error"):
            print(f"Error en la respuesta para NIF {nif}: {response.json()['error']}")
            if len(captchaIds) == 0:
                print("No quedan más captchaIds disponibles. Deteniendo el scraper.")
                break
            else:
                captchaId = captchaIds.pop(0)
                print(f"Cambiando a nuevo captchaId: {captchaId}")
        save_as_json(response.json(), f"{nif}.json", directory="../data/cache/registradores/")
        time.sleep(1)  # Esperar 5 segundos entre solicitudes para evitar sobrecargar el servidor
    else:
        print(f"Error al obtener datos para NIF {nif}: Código de estado {response.status_code}")

rm_jsons = os.listdir("../data/cache/registradores/")
accumulated_data = []
for rm_json in rm_jsons:
    with open(f"../data/cache/registradores/{rm_json}", 'r', encoding='utf-8') as f:
        data = json.load(f)
        if data['data']['recordsFiltered'] != 0:
            try:
                accumulated_data.append(
                    {
                        "nif": data['data']['data'][0]['nif'],
                        "denominacion": data['data']['data'][0]['denominacion']['denominacion'],
                        "cnae": data['data']['data'][0].get('cnae', {}).get('codigo', None),
                        "estado": data['data']['data'][0].get('estado', None),
                        "direccion": data['data']['data'][0].get('domicilioSocial', {}).get('direccion', None),
                        "municipio": data['data']['data'][0].get('domicilioSocial', {}).get('poblacion', None),
                        "provincia": data['data']['data'][0].get('domicilioSocial', {}).get('provincia', None),
                    }
                )
            except Exception as e:
                print(f"Error al procesar el JSON {rm_json}: {e}.\n{json.dumps(data, indent=2)}")
pd.DataFrame(accumulated_data).to_csv("../data/bronze/registradores/registradores_startups_certificadas.csv", index=False)

## CNAE 2025

Si bien las páginas anteriores ya nos proporcionan el dato del código CNAE de cada empresa, conviene poder agregar los códigos de forma consistente en secciones como lo hace el propio sistema de códigos CNAE del INE. Se Priorizará usar CNAE 2025 porque es el nuevo estándar que usa el INE.

La jerarquía a utilizar será la que aparece en este [documento pdf publicado por el INE](https://www.ine.es/normativa/leyes/cse/proyecto_CNAE2025.pdf). En dicho documento tambiém aparece la justificación del nuevo sistema que se fundamenta en la necesidad de realizar la adaptación nacional del NACE Rev.2.1 como lo recomienda la Comisión Europea y por modernizar el sistema de clasificación con más de 10 años de antigüedad.

Se ha encontrado los códigos completos en la siguiente [url](https://ine.es/dyngs/INEbase/es/operacion.htm?c=Estadistica_C&cid=1254736177032&menu=ultiDatos&idp=1254735976614) en formato hoja de calculo excel.

In [ ]:
from src.processing.ine_parsers import parse_cnae_2025_excel
# en el excel a procesar la primera hoja es una portada,
# la segunda hoja contiene la estructura CNAE 2025
# nos aseguramos de usar unicamente las columnas A y B
cnae_df = parse_cnae_2025_excel(
    excel_file_path="../data/bronze/INE/Estructura_CNAE2025.xlsx",
    sheet_name="Estructura_CNAE2025",
    usecols="A:B",
)

cnae_df.to_csv("../data/silver/cnae_2025.csv", sep=";", index=False)